## Predictive Alignment

- Input: $z_{pred}$, $z_{target}$
- Tools: CKA, subspace alignment, comparative probing

*"does the predictor's representation match the encoder's, and where does it diverge?"*

In [ ]:

import numpy as np
from pathlib import Path

from src.utils.io import (EXPERIMENTS_DIR, DATA_DIR, 
                          load_embeddings, load_sequences_dict, 
                          load_metadata, load_json)
from src.utils.seed import load_exp_seed, set_global_seed

In [ ]:
# -- Config Settings --
MODEL       = "test_01"
EMB_NAME    = "embeddings_40.npz"

In [ ]:
EXPERIMENTS     = Path("experiments")
MODEL_DIR       = EXPERIMENTS / MODEL
ANALYSIS_DIR    = MODEL_DIR / "analysis" / "prediction"
FIGURES_DIR     = ANALYSIS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
SEQUENCES_PATH  = Path(DATA_DIR / "sequences.jsonl")

set_global_seed(load_exp_seed(EXPERIMENTS_DIR / MODEL))

# -- Load embeddings --
emb, EMB_PATH = load_embeddings(MODEL_DIR, EMB_NAME)
print(f"Embeddings: {EMB_PATH.name}")

z_encs = emb["z_encs"]             # (N, C_padded, D)
z_pred = emb["z_pred"]             # (N, D)
z_target = emb["z_target"]         # (N, D)
ctx_pad_masks = emb["ctx_pad_masks"]  # (N, C_padded)
subject_ids = emb["subject_ids"]   # (N,)
mask_pos = emb["mask_pos"]         # (N,)
pred_error = z_pred - z_target     # (N, D)

# Flatten valid encounters from z_encs
valid = ~ctx_pad_masks.astype(bool)
z_enc_flat = z_encs[valid]         # (N_valid, D)
enc_subject_ids = np.broadcast_to(
    subject_ids[:, None], ctx_pad_masks.shape)[valid]
enc_positions = np.broadcast_to(
    np.arange(ctx_pad_masks.shape[1])[None, :], ctx_pad_masks.shape)[valid]

print(f"  z_encs:      {z_encs.shape}")
print(f"  z_enc_flat:  {z_enc_flat.shape}")
print(f"  z_pred:      {z_pred.shape}")
print(f"  z_target:    {z_target.shape}")
print(f"  pred_error:  {pred_error.shape}")
print(f"  subjects:    {len(np.unique(subject_ids))}")

# -- Load labels from sequences.jsonl
patients = load_sequences_dict(SEQUENCES_PATH)

unique_sids = np.unique(subject_ids)
label_escalation_patient = np.array([
    patients[sid].get("label_escalation", 0) for sid in unique_sids])
label_30d_patient = np.array([
    patients[sid].get("label_30d", 0) for sid in unique_sids])
 
# Sample-level encounter labels (for the masked encounter)
label_esc_per_sample = np.array([
    patients[str(sid)]["label_escalation_per_enc"][int(mp)]
    if "label_escalation_per_enc" in patients[str(sid)] else 0
    for sid, mp in zip(subject_ids, mask_pos)])
 
print(f"  Escalation rate (patient): {label_escalation_patient.mean():.3f}")
print(f"  Escalation rate (encounter): {label_esc_per_sample.mean():.3f}")
print(f"  30d readmit rate: {label_30d_patient.mean():.3f}")

In [ ]:
try:
    meta, meta_names, meta_pids = load_metadata(DATA_DIR)
    print(f"  Metadata: {meta.shape[0]} patients x {meta.shape[1]} features")
except FileNotFoundError:
    print("  Metadata not found")

In [ ]:
results = load_json(ANALYSIS_DIR / "representation.json")
if results is None:
    raise FileNotFoundError(f"No predictor results found")

## Plotting

In [ ]:
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")

**Probe comparison bar chart**: grouped bars: $z_{pred}$ vs $z_{target}$ AUROC for each label (escalation encounter-level, escalation patient-level, 30d readmission). Error bars from CV std.

This is the "information preservation" figure - where $z_{pred}$ matches $z_{target}$, the predictor learned the signal.

In [ ]:
from src.analysis.plotting import _s2_probe_comparison
_s2_probe_comparison(results, show=True, save=False, fig_dir=FIGURES_DIR)

**PCA subspace alignment**: bar chart of cosine of principal angles for top-k PCs. Values near 1.0 = aligned, near 0.0 = orthogonal. Annotate CKA score as text.

In [ ]:
from src.analysis.plotting import _s2_subspace_alignment
_s2_subspace_alignment(results, show=True, save=False, fig_dir=FIGURES_DIR)

**SAE vocabulary overlap** (if SAE matching stats exist): histogram of cosine similarities between matched $z_{pred}$ and $z_{target}$ dictionary directions. Vertical line at 0.8 threshold. Annotate fraction above threshold.

In [ ]:
from src.analysis.plotting import _s2_sae_vocab_overlap
_s2_sae_vocab_overlap(results, show=True, save=False, fig_dir=FIGURES_DIR)